**OBSOLETE -- superseded by the 2026-08-25 reorientation plan.**

Kept in the repo as a historical record (real, validated results at the
time -- e.g. the calibration submission that scored public LB 0.596) but
not maintained going forward. The gold+weak preprocessing/training/
inference pipeline built here is being rebuilt from scratch under the
new plan (measurement-gate fix, verified slice ordering, validated label
sets, a rebuilt `src/`-backed preprocessing pass, and a 6-slot
attention model), tracked in the `v2` notebooks
(`00v2_measurement_gate.ipynb`, `01v2_slice_ordering.ipynb`, ...). See
README.md for the current plan.

# 05b - Gold+weak training (Fase 5)

Consolidated final notebook (2026-08-23) for a **Save & Run All** background
commit. Cells 1-6 were built and individually validated interactively on
Kaggle in a previous session (see
`docs/superpowers/plans/2026-08-19-fase5-weak-training.md` for the
cell-by-cell validation log); the gold-only control arm's result from that
session (pooled OOF macro-AUC 0.5286, std 0.0113, 3 seeds) is reproduced by
this notebook, not hardcoded. Cells 8-10 (metrics aggregation, gate decision,
per-finding breakdown) are new here but only aggregate already-validated
per-run outputs using `src/evaluate.py`'s formulas, ported as-is.

**Goal:** does adding the 4,349 weak-labeled studies to training beat the
gold-only control arm's pooled OOF macro-ROC-AUC, under the identical
protocol (5-fold `GroupKFold`, pooled out-of-fold gold predictions, 3 seeds)?
See `docs/superpowers/specs/2026-08-19-fase5-weak-training-design.md` for the
full design and gate definition (Section 5).

**Expected runtime:** ~110s for `gold_only` (15 runs: 3 seeds x 5 folds) +
~1.8h for `gold_weak` (15 runs, ~438s/fold measured previously) = under 2h
total, run unattended as a Kaggle background commit (immune to the
browser-inactivity timeout that killed the previous interactive attempt).

Self-contained: no `from src import ...` (Kaggle does not mount this repo).

## Cell 1 - Imports, mount, constants

In [ ]:
# Self-contained (no `from src import ...`) -- Kaggle doesn't mount this repo.
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

RAW_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
TRIPLETS_DIR = Path("/kaggle/input/datasets/alherma7/triplets-knee")
assert RAW_DIR.exists(), f"Competition data not found at {RAW_DIR}"
assert TRIPLETS_DIR.exists(), f"Triplets dataset not found at {TRIPLETS_DIR}"

npy_files = list(TRIPLETS_DIR.glob("*.npy"))
print(f"Triplet files found: {len(npy_files)}")
assert len(npy_files) == 4407, f"Expected 4407 triplet files, found {len(npy_files)}"

RANDOM_STATE = 42
CV_FOLDS = 5
SEEDS = [42, 43, 44]
ARMS = ["gold_only", "gold_weak"]

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

FINDINGS = [
    "acl_injury", "mcl_injury", "medial_meniscus_tear", "lateral_meniscus_tear",
    "oa_medial_compartment", "oa_lateral_compartment", "oa_patellofemoral_compartment",
    "effusion", "synovitis", "bakers_cyst", "bone_contusion", "fracture",
]
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL", "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus", "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA", "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA", "effusion": "Effusion",
    "synovitis": "Synovitis", "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion", "fracture": "Fracture",
}
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("timm:", timm.__version__)

## Cell 2 - Label/fold table

Ports `src/labelers.py::label_report/label_reports/report_group_key` and
`src/data.py::load_training_labels` bodies (graduated 2026-08-18). Expected
invariants (validated last session against this exact logic): 4,407 rows,
58 gold, 54 duplicate report-template groups / 206 studies, 0 groups split
across folds. Gold-per-fold counts may differ slightly from the historical
16/11/14/8/9 (likely a `scikit-learn` version difference between
environments -- confirmed harmless last session, `GroupKFold` is
deterministic given identical input, so a differing valid partition doesn't
indicate a logic bug).

In [ ]:
# Ported from src/labelers.py (graduated 2026-08-18) -- same logic, self-contained.
import hashlib
import re
import unicodedata


def _normalize(text):
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = unicodedata.normalize("NFKD", t)
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    t = re.sub(r"[_\-/\\]+", " ", t)
    t = re.sub(r"[ \t]+", " ", t)
    return t


_BULLET_PREFIX_RE = re.compile(r"^[>*\u2022\-]+\s*")


def _unwrap(text):
    if not isinstance(text, str):
        return ""
    out = []
    for line in text.split("\n"):
        s = line.strip()
        s_check = _BULLET_PREFIX_RE.sub("", s)
        if (out and out[-1] and not re.search(r"[.;:!?>*\u2022]$", out[-1])
                and len(out[-1].split()) >= 4 and s_check and not s_check[:1].isupper()):
            out[-1] = out[-1] + " " + s
        else:
            out.append(s)
    return "\n".join(out)


_SENT_SPLIT_RE = re.compile(r"(?<=[.;!?])\s*|\n+")


def _clauses(text):
    norm = _normalize(_unwrap(text))
    return [c.strip() for c in _SENT_SPLIT_RE.split(norm) if c and c.strip()]


_SUB_CLAUSE_SPLIT_RE = re.compile(r"\s+but\s+|\s+pero\s+|\s+aunque\s+|\s+although\s+")


def _sub_clauses(clause):
    return [s.strip() for s in _SUB_CLAUSE_SPLIT_RE.split(clause) if s and s.strip()]


_NEGATION_SCOPE_CUES = [
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\babsent\b",
    r"no evidence of", r"no sign", r"negative for",
    r"\bsin\b", r"\bausen", r"\bninguna\b", r"\bnegativ",
    r"no se (?:observa|evidencia|aprecia)",
]
_NEGATION_PREDICATE_CUES = [
    r"\bnormal\b", r"\bintact\b", r"\bunremarkable\b", r"within normal limit",
    r"dentro de (los )?l[i0]mites normales",
]
_NEGATION_SCOPE_RE = re.compile("|".join(_NEGATION_SCOPE_CUES))
_NEGATION_PREDICATE_RE = re.compile("|".join(_NEGATION_PREDICATE_CUES))


def _negation_applies(sub_clause, anatomy_re, pathology_re):
    for m in _NEGATION_SCOPE_RE.finditer(sub_clause):
        arg = sub_clause[m.end():]
        if anatomy_re.search(arg) or pathology_re.search(arg):
            return True
    for m in _NEGATION_PREDICATE_RE.finditer(sub_clause):
        arg = sub_clause[:m.start()]
        if anatomy_re.search(arg) or pathology_re.search(arg):
            return True
    return False


_OA_PATHOLOGY_CUES = [
    r"osteoarthrit", r"osteoarthros", r"osteoartr", r"chondrosis",
    r"(cartilage|chondral) (loss|thinning|defect|fissur)",
    r"joint space narrowing", r"osteophyte", r"osteofito", r"spurring",
    r"pinzamiento", r"degenerat", r"adelgazamiento del cartilago",
    r"chondromalacia", r"condromalacia", r"subchondral cystic",
]

FINDING_LEXICON = {
    "acl_injury": dict(
        anatomy=[r"\bacl\b", r"anterior cruciate ligament", r"ligamento cruzado anterior", r"\blca\b"],
        pathology=[r"\btear", r"\btorn\b", r"ruptur", r"sprain", r"rotur", r"desgarr", r"esguinc", r"discontinuit"],
    ),
    "mcl_injury": dict(
        anatomy=[r"\bmcl\b", r"medial collateral ligament", r"ligamento colateral medial", r"ligamento lateral interno"],
        pathology=[r"\btear", r"\btorn\b", r"ruptur", r"sprain", r"rotur", r"desgarr", r"esguinc"],
    ),
    "medial_meniscus_tear": dict(
        anatomy=[r"medial meniscus", r"menisco medial"],
        pathology=[r"\btear", r"\btorn\b", r"rotur", r"desgarr", r"extrusion", r"extrusi[o0]n", r"macerat"],
    ),
    "lateral_meniscus_tear": dict(
        anatomy=[r"lateral meniscus", r"menisco lateral"],
        pathology=[r"\btear", r"\btorn\b", r"rotur", r"desgarr", r"extrusion", r"extrusi[o0]n", r"macerat"],
    ),
    "oa_medial_compartment": dict(
        anatomy=[r"medial compartment", r"medial femorotibial", r"medial femoral condyle",
                 r"medial tibial plateau", r"compartimento medial"],
        pathology=_OA_PATHOLOGY_CUES,
    ),
    "oa_lateral_compartment": dict(
        anatomy=[r"lateral compartment", r"lateral femorotibial", r"lateral femoral condyle",
                 r"lateral tibial plateau", r"compartimento lateral"],
        pathology=_OA_PATHOLOGY_CUES,
    ),
    "oa_patellofemoral_compartment": dict(
        anatomy=[r"patellofemoral", r"femoropatelar", r"femororrotulian", r"patelofemoral"],
        pathology=_OA_PATHOLOGY_CUES,
    ),
    "effusion": dict(
        anatomy=[r"\beffusion\b", r"derrame articular", r"\bderrame\b", r"efusi[o0]n"],
        pathology=[r"\beffusion\b", r"derrame", r"efusi[o0]n", r"fluid collection", r"joint fluid"],
    ),
    "synovitis": dict(
        anatomy=[r"synovit", r"sinovit", r"synovial (thickening|hypertrophy|proliferation)", r"engrosamiento sinovial"],
        pathology=[r"synovit", r"sinovit", r"synovial (thickening|hypertrophy|proliferation)", r"engrosamiento sinovial"],
    ),
    "bakers_cyst": dict(
        anatomy=[r"baker'?s? cyst", r"popliteal cyst", r"quiste de baker", r"quiste poplite"],
        pathology=[r"baker'?s? cyst", r"popliteal cyst", r"quiste de baker", r"quiste poplite"],
    ),
    "bone_contusion": dict(
        anatomy=[r"bone (marrow )?contusion", r"bone (marrow )?edema", r"contusi[o0]n [o0]sea", r"edema [o0]seo", r"edema medular"],
        pathology=[r"bone (marrow )?contusion", r"bone (marrow )?edema", r"contusi[o0]n [o0]sea", r"edema [o0]seo", r"edema medular"],
    ),
    "fracture": dict(
        anatomy=[r"\bfractur"],
        pathology=[r"\bfractur", r"cortical (break|disruption)", r"trabecular fracture"],
    ),
}

_COMPILED_LEXICON = {
    finding: (re.compile("|".join(cues["anatomy"])), re.compile("|".join(cues["pathology"])))
    for finding, cues in FINDING_LEXICON.items()
}


def label_report(report_text, finding):
    anatomy_re, pathology_re = _COMPILED_LEXICON[finding]
    votes = []
    for clause in _clauses(report_text):
        for sub in _sub_clauses(clause):
            if not anatomy_re.search(sub):
                continue
            if _negation_applies(sub, anatomy_re, pathology_re):
                votes.append(0.0)
            elif pathology_re.search(sub):
                votes.append(1.0)
    if not votes:
        return 0.5
    return 1.0 if max(votes) == 1.0 else 0.0


def label_reports(reports, findings):
    if "StudyInstanceUID" in reports.columns:
        reports = reports.set_index("StudyInstanceUID")
    return pd.DataFrame({
        finding: reports["Report"].apply(lambda text: label_report(text, finding))
        for finding in findings
    }, index=reports.index)


def report_group_key(report_text):
    if not isinstance(report_text, str):
        normalized = ""
    else:
        t = unicodedata.normalize("NFKD", report_text.lower())
        t = "".join(ch for ch in t if not unicodedata.combining(ch))
        normalized = re.sub(r"\s+", " ", t).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


# --- load_training_labels (ported from src/data.py) ---
train = pd.read_csv(RAW_DIR / "train.csv")
reports = train[["StudyInstanceUID", "Report"]].set_index("StudyInstanceUID")

gold_mask = train[LABEL_COLS].notna().all(axis=1)
gold = train.loc[gold_mask, ["StudyInstanceUID"] + LABEL_COLS].set_index("StudyInstanceUID")
gold.columns = FINDINGS
assert len(gold) == 58, f"Expected 58 gold studies, found {len(gold)}"

is_gold = reports.index.isin(gold.index)
weak_reports = reports.loc[~is_gold].reset_index()
weak_labels = label_reports(weak_reports, FINDINGS)

combined = pd.concat([gold[FINDINGS], weak_labels[FINDINGS]])
combined = combined.loc[reports.index]
combined["is_gold"] = is_gold

group_keys = reports["Report"].apply(report_group_key)
gkf = GroupKFold(n_splits=CV_FOLDS)
fold = pd.Series(-1, index=reports.index, dtype=int)
for fold_idx, (_, val_idx) in enumerate(gkf.split(reports, groups=group_keys.to_numpy())):
    fold.iloc[val_idx] = fold_idx
combined["fold"] = fold

label_table = combined
print(f"label_table: {len(label_table)} rows, {int(label_table['is_gold'].sum())} gold")
print("Gold per fold:")
print(label_table.loc[label_table["is_gold"], "fold"].value_counts().sort_index())

_group_counts = group_keys.value_counts()
n_dup_groups = int((_group_counts > 1).sum())
n_dup_studies = int(group_keys.map(_group_counts).gt(1).sum())
print(f"Duplicate report-template groups: {n_dup_groups} ({n_dup_studies} studies)")

split_across = 0
for _key, _idxs in group_keys.groupby(group_keys).groups.items():
    if label_table.loc[_idxs, "fold"].nunique() > 1:
        split_across += 1
print(f"Groups split across folds: {split_across}")

## Cell 3 - Dataset / DataLoader

Lazily loads each study's `.npy` triplet from the mounted dataset on
`__getitem__` (rather than preloading all ~7.3GB into RAM up front) so
`DataLoader(num_workers=...)` can hide disk I/O behind GPU compute.
Studies present in `label_table` but missing a triplet file are excluded
(0 expected -- Notebook A reported 0/4,407 preprocessing failures).

In [ ]:
class TripletDataset(Dataset):
    def __init__(self, study_ids, label_table, triplets_dir):
        self.study_ids = list(study_ids)
        self.label_table = label_table
        self.triplets_dir = triplets_dir

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, idx):
        study_id = self.study_ids[idx]
        triplet = np.load(self.triplets_dir / f"{study_id}.npy")
        target = self.label_table.loc[study_id, FINDINGS].to_numpy(dtype=np.float32)
        is_gold = bool(self.label_table.loc[study_id, "is_gold"])
        return torch.from_numpy(triplet), torch.from_numpy(target), is_gold, study_id


available_ids = {p.stem for p in npy_files}
missing = set(label_table.index) - available_ids
print(f"Studies missing a triplet file: {len(missing)}")
label_table = label_table.loc[label_table.index.isin(available_ids)]
print(f"label_table after intersection: {len(label_table)} rows")

smoke_ds = TripletDataset(label_table.index[:2], label_table, TRIPLETS_DIR)
x0, y0, g0, id0 = smoke_ds[0]
print(f"Smoke test: triplet shape={tuple(x0.shape)}, dtype={x0.dtype}, target shape={tuple(y0.shape)}, is_gold={g0}")
print(f"Target values: {y0.numpy()}")

## Cell 4 - Model (EfficientNet-B0 + differential-LR head, unchanged from Fase 4)

In [ ]:
class BaselineFindingModel(nn.Module):
    def __init__(self, backbone_name, n_findings, dropout=0.5, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.backbone.num_features, n_findings),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


def differential_lr_param_groups(model, backbone_lr, head_lr):
    return [
        {"params": list(model.backbone.parameters()), "lr": backbone_lr},
        {"params": list(model.head.parameters()), "lr": head_lr},
    ]


_sanity_model = BaselineFindingModel("efficientnet_b0", n_findings=len(FINDINGS), pretrained=True)
_sanity_groups = differential_lr_param_groups(_sanity_model, backbone_lr=1e-5, head_lr=1e-3)
n_backbone = sum(p.numel() for p in _sanity_groups[0]["params"])
n_head = sum(p.numel() for p in _sanity_groups[1]["params"])
print(f"Backbone params: {n_backbone:,} (lr={_sanity_groups[0]['lr']})")
print(f"Head params: {n_head:,} (lr={_sanity_groups[1]['lr']})")
del _sanity_model, _sanity_groups

## Cell 5 - Loss with corrected `pos_weight`

`pos_weight = n_negative / n_positive` per finding, counted **only over
hard-labeled rows** (`target != 0.5`) in the training fold (spec Section 4
-- the first spec draft had this inverted, fixed in review). Soft targets
(0.0/0.5/1.0) still go into `BCEWithLogitsLoss` as-is for every row -- this
only changes how the imbalance correction is *counted*, not which rows
contribute to the loss.

In [ ]:
def compute_pos_weight(targets_df):
    """n_negative / n_positive per finding, over hard-labeled rows only."""
    weights = []
    for col in FINDINGS:
        hard = targets_df[col][targets_df[col] != 0.5]
        n_pos = max((hard == 1.0).sum(), 1)
        n_neg = (hard == 0.0).sum()
        weights.append(n_neg / n_pos)
    return torch.tensor(weights, dtype=torch.float32)


_check = compute_pos_weight(label_table[FINDINGS])
for finding, w in zip(FINDINGS, _check.tolist()):
    print(f"  {finding}: {w:.2f}")
print()
print("oa_lateral_compartment pos_weight (sanity: should reflect real class balance among")
print("hard-labeled rows, not collapse toward 1.0 from abstention dilution):",
      f"{_check[FINDINGS.index('oa_lateral_compartment')]:.2f}")

## Cell 6 - Single (arm, seed, fold) training function

`batch_size=8` for `gold_only` (matches Fase 4, ~44-50 studies/fold --
`batch_size=48` was tried first and collapsed this arm to ~1 gradient
step/epoch, severe underfitting); `batch_size=48` for `gold_weak`
(~3,500-4,400 studies/fold, confirmed to fit a T4's 16GB with ~3.3GB
headroom). Fixed 8-epoch budget, no val-based checkpoint selection --
predictions are always from the final epoch (Fase 4 audit fix (2): the
original 0.574 result was inflated by selecting the best epoch against the
same val data used to report it).

`macro_roc_auc_hard` is a train-only diagnostic: `y_true` for `gold_weak`'s
training rows contains soft 0.5 targets that `roc_auc_score` can't score,
so it's restricted to hard-labeled cells for a well-defined number (spec
Section 4's diagnostic-logging requirement). The real gate metric (Cell 8)
always scores gold-only validation rows, which are always hard-labeled, so
it uses the plain `macro_roc_auc`/`per_finding_roc_auc` (same formula as
`src/evaluate.py`, unaffected by this distinction).

In [ ]:
def macro_roc_auc_hard(y_true_df, y_pred_df):
    aucs = []
    for col in y_true_df.columns:
        mask = y_true_df[col] != 0.5
        if mask.sum() == 0 or y_true_df.loc[mask, col].nunique() < 2:
            continue
        aucs.append(roc_auc_score(y_true_df.loc[mask, col], y_pred_df.loc[mask, col]))
    return float(np.mean(aucs)) if aucs else float("nan")


def macro_roc_auc(y_true_df, y_pred_df):
    aucs = [roc_auc_score(y_true_df[col], y_pred_df[col]) for col in y_true_df.columns]
    return float(np.mean(aucs))


def per_finding_roc_auc(y_true_df, y_pred_df):
    return pd.Series({col: roc_auc_score(y_true_df[col], y_pred_df[col]) for col in y_true_df.columns})


N_EPOCHS = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def run_arm_seed_fold(arm, seed, fold_idx, label_table, triplets_dir, n_epochs=N_EPOCHS, verbose=True):
    torch.manual_seed(seed)

    train_mask = label_table["fold"] != fold_idx
    if arm == "gold_only":
        train_mask = train_mask & label_table["is_gold"]
    train_ids = label_table.index[train_mask]

    val_mask = (label_table["fold"] == fold_idx) & label_table["is_gold"]
    val_ids = label_table.index[val_mask]

    batch_size = 8 if arm == "gold_only" else 48
    eval_batch_size = 64

    train_ds = TripletDataset(train_ids, label_table, triplets_dir)
    val_ds = TripletDataset(val_ids, label_table, triplets_dir)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    train_eval_loader = DataLoader(train_ds, batch_size=eval_batch_size, shuffle=False, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=eval_batch_size, shuffle=False, num_workers=2, pin_memory=True)

    train_pos_weight = compute_pos_weight(label_table.loc[train_ids, FINDINGS]).to(DEVICE)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=train_pos_weight)

    model = BaselineFindingModel("efficientnet_b0", n_findings=len(FINDINGS), pretrained=True).to(DEVICE)
    optimizer = torch.optim.Adam(
        differential_lr_param_groups(model, backbone_lr=1e-5, head_lr=1e-3),
        weight_decay=1e-2,
    )

    for epoch in range(n_epochs):
        model.train()
        epoch_loss, n_seen = 0.0, 0
        for xb, yb, _, _ in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
            n_seen += len(xb)
        if verbose:
            print(f"    [{arm}/seed{seed}/fold{fold_idx}] epoch {epoch+1}/{n_epochs}: loss={epoch_loss/n_seen:.4f}")

    # Final-epoch predictions, no val-based checkpoint selection (Fase 4 audit fix (2)).
    model.eval()

    def predict(loader):
        preds, ids = [], []
        with torch.no_grad():
            for xb, _, _, batch_ids in loader:
                xb = xb.to(DEVICE)
                probs = torch.sigmoid(model(xb)).cpu().numpy()
                preds.append(probs)
                ids.extend(batch_ids)
        return pd.DataFrame(np.concatenate(preds), index=ids, columns=FINDINGS)

    train_preds = predict(train_eval_loader)
    val_preds = predict(val_loader)

    train_true = label_table.loc[train_preds.index, FINDINGS]
    train_auc = macro_roc_auc_hard(train_true, train_preds)

    del model
    torch.cuda.empty_cache()

    return train_preds, val_preds, train_auc


# Smoke test (1 epoch, gold_only, fold 0) -- cheap sanity check before the full 30-run loop.
_t0 = time.time()
_train_p, _val_p, _train_auc = run_arm_seed_fold(
    "gold_only", seed=42, fold_idx=0, label_table=label_table, triplets_dir=TRIPLETS_DIR,
    n_epochs=1, verbose=True,
)
print(f"Smoke test done in {time.time() - _t0:.1f}s: val_ids={len(_val_p)}, train_auc={_train_auc:.4f}")

## Cell 7 - Outer loop: 2 arms x 3 seeds x 5 folds (30 runs)

Pools each (arm, seed)'s 5-fold gold-only OOF predictions into one 58-row
table before scoring (spec Section 5 -- avoids the small-fold single-class
failure mode that a per-fold `macro_roc_auc` risks for rare findings like
MCL). This is the long-running cell -- expected ~110s for `gold_only` +
~1.8h for `gold_weak`, unattended under Save & Run All.

In [ ]:
results = {}
timings = {}

for arm in ARMS:
    for seed in SEEDS:
        t0 = time.time()
        fold_val_preds, fold_train_aucs = [], []
        for fold_idx in range(CV_FOLDS):
            _, val_preds, train_auc = run_arm_seed_fold(
                arm, seed, fold_idx, label_table, TRIPLETS_DIR, n_epochs=N_EPOCHS, verbose=False,
            )
            fold_val_preds.append(val_preds)
            fold_train_aucs.append(train_auc)
            print(f"  {arm}/seed={seed}/fold={fold_idx}: val_n={len(val_preds)} train_auc={train_auc:.4f}")

        val_pooled = pd.concat(fold_val_preds)
        assert len(val_pooled) == 58, f"{arm}/{seed}: expected 58 pooled gold val rows, got {len(val_pooled)}"
        assert val_pooled.index.is_unique, f"{arm}/{seed}: duplicate study in pooled val predictions"

        results[(arm, seed)] = dict(val_preds=val_pooled, train_auc_mean=float(np.mean(fold_train_aucs)))
        timings[(arm, seed)] = time.time() - t0
        print(f"{arm}/seed={seed}: done in {timings[(arm, seed)]:.1f}s, "
              f"train_auc_mean={results[(arm, seed)]['train_auc_mean']:.4f}\n")

print("All 30 (arm, seed, fold) runs complete.")
print(f"Total wall time: {sum(timings.values()) / 60:.1f} min")

## Cell 8 - Compute metrics

`macro_roc_auc`/`per_finding_roc_auc` on the pooled 58-row gold val table,
per (arm, seed) -- same formula as `src/evaluate.py`, reused as-is (gold
val rows are always hard-labeled, so the soft-label distinction from Cell 6
doesn't apply here). Aggregated to mean +/- std across the 3 seeds, per
arm.

In [ ]:
rows = []
for (arm, seed), r in results.items():
    val_true = label_table.loc[r["val_preds"].index, FINDINGS]
    val_macro = macro_roc_auc(val_true, r["val_preds"])
    rows.append(dict(arm=arm, seed=seed, val_macro_auc=val_macro, train_macro_auc=r["train_auc_mean"]))

metrics_df = pd.DataFrame(rows)
print(metrics_df)

summary = metrics_df.groupby("arm")[["val_macro_auc", "train_macro_auc"]].agg(["mean", "std"])
print()
print("Per-arm summary (mean +/- std across 3 seeds):")
print(summary)

## Cell 9 - Gate decision

The gate (spec Section 5): `gold_weak`'s pooled OOF macro-AUC must beat
**this run's own `gold_only` control arm**, not Fase 4's raw 0.532 (which
used a different CV protocol -- shown below for context only).

In [ ]:
gold_only_mean = summary.loc["gold_only", ("val_macro_auc", "mean")]
gold_only_std = summary.loc["gold_only", ("val_macro_auc", "std")]
gold_weak_mean = summary.loc["gold_weak", ("val_macro_auc", "mean")]
gold_weak_std = summary.loc["gold_weak", ("val_macro_auc", "std")]

print(f"gold-only control arm: {gold_only_mean:.4f} (std {gold_only_std:.4f})")
print(f"gold+weak arm:         {gold_weak_mean:.4f} (std {gold_weak_std:.4f})")
print("Fase 4 historical (different protocol, context only): 0.532 (std 0.021)")
print()
if gold_weak_mean > gold_only_mean:
    print(f"WIN: gold+weak ({gold_weak_mean:.4f}) beats the gold-only control ({gold_only_mean:.4f}) "
          f"by {gold_weak_mean - gold_only_mean:.4f}.")
else:
    print(f"NO WIN: gold+weak ({gold_weak_mean:.4f}) does not beat the gold-only control ({gold_only_mean:.4f}) "
          f"(delta {gold_weak_mean - gold_only_mean:.4f}).")

## Cell 10 - Per-finding breakdown + train/val gap

Inspects whether specific findings (e.g. `oa_lateral_compartment`, the
weakest-labeled per Fase 3) drag `gold_weak` down disproportionately (spec
Section 8 open risk), and whether the train/val AUC gap narrowed relative
to Fase 4's severe 0.95-0.98 train-AUC overfitting regime.

In [ ]:
per_finding_rows = []
for (arm, seed), r in results.items():
    val_true = label_table.loc[r["val_preds"].index, FINDINGS]
    pf = per_finding_roc_auc(val_true, r["val_preds"])
    pf["arm"] = arm
    pf["seed"] = seed
    per_finding_rows.append(pf)

per_finding_df = pd.DataFrame(per_finding_rows)
per_finding_summary = per_finding_df.groupby("arm")[FINDINGS].mean().T
print("Per-finding mean val AUC (across 3 seeds), by arm:")
print(per_finding_summary.sort_values(per_finding_summary.columns[0]))

print()
print("Train/val macro-AUC gap per arm (mean across seeds):")
gap = summary[("train_macro_auc", "mean")] - summary[("val_macro_auc", "mean")]
print(gap)

## Closing task (after this notebook finishes)

Update `README.md`'s Progress log with the real result -- win or negative
result, per this project's logging convention. Include: the gold-only
control arm's number, the gold+weak arm's number, per-finding breakdown
highlights, and the train/val AUC gap finding. If it's a win, open the
follow-up conversation about graduating the image pipeline to `src/`
(spec Section 1, Section 5) -- a separate step, not part of this
notebook.